# FedX-PALM: Centralized YOLOv11 Training (Baseline)

**Master's Thesis - Rachma Dianty (201012420007)** · Telkom University, 2026

---

## Tujuan
Notebook ini melatih YOLOv11 secara **centralized** (semua data digabung di satu tempat) menggunakan dataset Roboflow Anda. Ini adalah **baseline pembanding** untuk eksperimen Federated Learning di tahap berikutnya (Tabel 4.4 tesis: target centralized mAP@0.5 ≈ 0.921).

## Tahapan
1. Setup environment (Ultralytics + Roboflow)
2. Verifikasi class names dari Roboflow (CRITICAL — pastikan urutan sesuai tesis)
3. Download dataset (6 kelas)
4. Train YOLOv11 Nano (100 epochs, batch 16, img 640)
5. Evaluasi mAP@0.5 / Precision / Recall / F1
6. Simpan checkpoint `.pt` ke Google Drive (untuk dipakai sebagai $w_0$ di FL nanti)

## Spesifikasi (Tabel 3.5 tesis)
| Param | Nilai |
|-------|-------|
| Model | YOLOv11 Nano (`yolo11n.pt`) |
| Classes | 6 |
| Input | 640×640 |
| Optimizer | AdamW |
| Loss | CIoU |
| LR | 0.01 |
| Batch | 16 |
| Epochs | 100 |
| GPU | NVIDIA Tesla T4 (Colab Pro) |

## 1. Setup Environment

In [1]:
!pip install -q ultralytics roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 105.5 MB/s eta 0:00:00


In [ ]:
import torch, ultralytics, roboflow
from ultralytics import YOLO

print(f'PyTorch     : {torch.__version__}')
print(f'CUDA        : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU         : {torch.cuda.get_device_name(0)}')
    print(f'VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Ultralytics : {ultralytics.__version__}')
print(f'Roboflow    : {roboflow.__version__}')

## 2. Verifikasi Class Names dari Roboflow

**PENTING:** Cek nama & urutan kelas. Dataset Roboflow Anda pakai urutan alfabetis:

| Idx | Kelas (Roboflow) |
|-----|------------------|
| 0 | Abnormal |
| 1 | Empty Bunch |
| 2 | Overripe |
| 3 | Ripe |
| 4 | Underripe |
| 5 | Unripe |

Narasi tesis Bab 2.1.2 & Tabel 3.3/4.7/4.8 perlu disesuaikan ke urutan ini (lihat catatan revisi).

In [2]:
from roboflow import Roboflow

ROBOFLOW_API_KEY = 'Ej0bSMpeSri3ky0IYkOU'

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace('dydy-worker').project('palm-fruit-ripeness-detection-f6sac-ccb2z')

print(f'Project Name : {project.name}')
print(f'Type         : {project.type}')
print(f'Classes      : {project.classes}')
print()

# Roboflow uses alphabetical class order (decided: thesis follows this)
EXPECTED_ROBOFLOW = ['Abnormal', 'Empty Bunch', 'Overripe', 'Ripe', 'Underripe', 'Unripe']
roboflow_classes = list(project.classes.keys()) if isinstance(project.classes, dict) else list(project.classes)

print(f'Expected (Roboflow alphabetical): {EXPECTED_ROBOFLOW}')
print(f'Actual (from Roboflow API)     : {roboflow_classes}')
print()

norm_expected = [c.lower().replace('_', ' ').strip() for c in EXPECTED_ROBOFLOW]
norm_actual   = [c.lower().replace('_', ' ').strip() for c in roboflow_classes]

if norm_actual == norm_expected:
    print('✅ MATCH: urutan kelas sesuai. Lanjut ke download dataset.')
elif sorted(norm_actual) == sorted(norm_expected):
    print('⚠️  Nama kelas sama, urutan beda. Lanjut, tapi catat urutan aktual untuk update kode/tesis.')
else:
    print('❌ CLASS MISMATCH: ada nama kelas yang berbeda — cek manual!')
    print('    Diff:', set(EXPECTED_ROBOFLOW) ^ set(roboflow_classes))

loading Roboflow workspace...
loading Roboflow project...
Project Name : Palm Fruit Ripeness Detection
Type         : object-detection
Classes      : {'Overripe': 2641, 'Empty Bunch': 857, 'Underripe': 2575, 'Ripe': 2973, 'Unripe': 2913, 'Abnormal': 2599}

Expected (Roboflow alphabetical): ['Abnormal', 'Empty Bunch', 'Overripe', 'Ripe', 'Underripe', 'Unripe']
Actual (from Roboflow API)     : ['Overripe', 'Empty Bunch', 'Underripe', 'Ripe', 'Unripe', 'Abnormal']

⚠️  Nama kelas sama, urutan beda. Lanjut, tapi catat urutan aktual untuk update kode/tesis.


## 3. Download Dataset

In [3]:
import os
os.makedirs('/content/data', exist_ok=True)

version = project.version(2)
dataset = version.download('yolov11', location='/content/data/palm_v2')
print(f'\nDataset location: {dataset.location}')
print(f'data.yaml       : {dataset.location}/data.yaml')


Extracting Dataset Version Zip to /content/data/palm_v2 in yolov11:: 100%|██████████| 21640/21640 [00:04<00:00, 4768.43it/s]



Dataset location: /content/data/palm_v2
data.yaml       : /content/data/palm_v2/data.yaml


In [ ]:
# Inspect data.yaml from Roboflow
with open(f'{dataset.location}/data.yaml') as f:
    print(f.read())

In [ ]:
# Count images per split & per class
from pathlib import Path
from collections import Counter

root = Path(dataset.location)
for split in ['train', 'valid', 'test']:
    img_dir = root / split / 'images'
    lbl_dir = root / split / 'labels'
    if not img_dir.exists():
        continue
    n_img = len(list(img_dir.glob('*.jpg'))) + len(list(img_dir.glob('*.png')))
    cls_counter = Counter()
    for lbl in lbl_dir.glob('*.txt'):
        with open(lbl) as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    cls_counter[int(parts[0])] += 1
    print(f'\n{split.upper():6s} | images={n_img:5d} | annotations per class: {dict(sorted(cls_counter.items()))}')

## 4. Train YOLOv11 Nano (Centralized Baseline)

Estimasi waktu di Tesla T4: ~5–7 menit per epoch × 100 epoch ≈ 8–12 jam.
Untuk smoke test pakai `epochs=10` dulu, baru `epochs=100` untuk run final.

In [ ]:
# ====== TRAINING CONFIG (sinkron dengan Tabel 3.5 tesis) ======
MODEL_VARIANT  = 'yolo11n.pt'   # Nano (smallest, fastest)
IMG_SIZE       = 640
BATCH_SIZE     = 16
EPOCHS         = 100            # ganti ke 10 untuk smoke test
OPTIMIZER      = 'AdamW'
LR0            = 0.01
PATIENCE       = 20
PROJECT_DIR    = '/content/runs/centralized'
RUN_NAME       = 'palm_yolov11n_100ep'

DATA_YAML = f'{dataset.location}/data.yaml'
print(f'Training config:')
print(f'  model     = {MODEL_VARIANT}')
print(f'  epochs    = {EPOCHS}')
print(f'  batch     = {BATCH_SIZE}')
print(f'  img       = {IMG_SIZE}')
print(f'  optimizer = {OPTIMIZER}, lr0={LR0}')
print(f'  data.yaml = {DATA_YAML}')

In [ ]:
model = YOLO(MODEL_VARIANT)

results = model.train(
    data       = DATA_YAML,
    epochs     = EPOCHS,
    imgsz      = IMG_SIZE,
    batch      = BATCH_SIZE,
    optimizer  = OPTIMIZER,
    lr0        = LR0,
    patience   = PATIENCE,
    project    = PROJECT_DIR,
    name       = RUN_NAME,
    exist_ok   = True,
    plots      = True,
    save       = True,
    device     = 0,
)
print('\n✅ Training selesai.')
print(f'Best weights: {PROJECT_DIR}/{RUN_NAME}/weights/best.pt')

## 5. Evaluasi Final pada Test Set

Metrik di sini akan jadi **angka real untuk Tabel 4.4 (baseline centralized)** di tesis Anda.

In [ ]:
best_weights = f'{PROJECT_DIR}/{RUN_NAME}/weights/best.pt'
best_model = YOLO(best_weights)

metrics = best_model.val(
    data  = DATA_YAML,
    imgsz = IMG_SIZE,
    batch = BATCH_SIZE,
    split = 'val',
    project = PROJECT_DIR,
    name    = f'{RUN_NAME}_eval',
    exist_ok = True,
)

print('\n=== Centralized Baseline Metrics ===')
print(f'mAP@0.5      : {metrics.box.map50:.4f}')
print(f'mAP@0.5:0.95 : {metrics.box.map:.4f}')
print(f'Precision    : {metrics.box.mp:.4f}')
print(f'Recall       : {metrics.box.mr:.4f}')

# Per-class
print('\n--- Per-class mAP@0.5 ---')
for i, name in enumerate(best_model.names.values()):
    if i < len(metrics.box.maps):
        print(f'  {i} {name:15s}: {metrics.box.maps[i]:.4f}')

In [ ]:
# Compute F1 (harmonic mean of precision & recall)
P, R = metrics.box.mp, metrics.box.mr
F1 = 2 * P * R / (P + R) if (P + R) > 0 else 0
print(f'F1-Score     : {F1:.4f}')

# Save summary to JSON for the thesis report
import json, datetime
summary = {
    'experiment'  : 'centralized_baseline',
    'timestamp'   : datetime.datetime.utcnow().isoformat(),
    'model'       : MODEL_VARIANT,
    'epochs'      : EPOCHS,
    'batch'       : BATCH_SIZE,
    'img_size'    : IMG_SIZE,
    'optimizer'   : OPTIMIZER,
    'lr0'         : LR0,
    'mAP_50'      : float(metrics.box.map50),
    'mAP_50_95'   : float(metrics.box.map),
    'precision'   : float(P),
    'recall'      : float(R),
    'f1'          : float(F1),
    'per_class_mAP50': {name: float(metrics.box.maps[i]) for i, name in enumerate(best_model.names.values()) if i < len(metrics.box.maps)},
}
with open(f'{PROJECT_DIR}/{RUN_NAME}/summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

## 6. Confusion Matrix & Sample Predictions

In [ ]:
from IPython.display import Image, display
import os

eval_dir = f'{PROJECT_DIR}/{RUN_NAME}_eval'
for fname in ['confusion_matrix.png', 'confusion_matrix_normalized.png', 'PR_curve.png', 'F1_curve.png']:
    fpath = os.path.join(eval_dir, fname)
    if os.path.exists(fpath):
        print(f'\n{fname}:')
        display(Image(fpath))

## 7. Simpan Checkpoint ke Google Drive

Hasil training disimpan ke Drive sehingga bisa dipakai sebagai $w_0$ (bobot awal) saat tahap Federated Learning di notebook berikutnya.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil
from pathlib import Path

drive_dir = Path('/content/drive/MyDrive/fedx-palm/centralized_baseline')
drive_dir.mkdir(parents=True, exist_ok=True)

# Copy best.pt + last.pt + summary.json
src_dir = Path(f'{PROJECT_DIR}/{RUN_NAME}')
for fname in ['weights/best.pt', 'weights/last.pt', 'summary.json', 'results.csv', 'args.yaml']:
    src = src_dir / fname
    if src.exists():
        dst = drive_dir / src.name
        shutil.copy2(src, dst)
        print(f'✓ {dst}')

# Copy eval plots
eval_src = Path(f'{PROJECT_DIR}/{RUN_NAME}_eval')
(drive_dir / 'eval_plots').mkdir(exist_ok=True)
for fname in ['confusion_matrix.png', 'confusion_matrix_normalized.png', 'PR_curve.png', 'F1_curve.png', 'P_curve.png', 'R_curve.png']:
    src = eval_src / fname
    if src.exists():
        shutil.copy2(src, drive_dir / 'eval_plots' / fname)
        print(f'✓ eval_plots/{fname}')

print(f'\nSemua artefak baseline tersimpan di: {drive_dir}')

## 8. Selesai — Langkah Berikutnya

Setelah baseline centralized ini berhasil:

1. **Catat angka final** untuk Tabel 4.4 tesis: mAP@0.5, Precision, Recall, F1 (centralized)
2. **Simpan `best.pt`** — ini jadi $w_0$ untuk semua client di tahap FL
3. **Notebook berikutnya:** `fedx_palm_colab_client.ipynb` untuk simulasi FL 4 client + DP-SGD + Grad-CAM++
4. **Setup VPS** (saat Anda sudah punya) untuk jalankan FL server aggregator — lihat panduan di `README.md`

**Catatan:** Angka centralized di sini akan jadi *upper bound* performa. Federated learning dengan Non-IID + DP umumnya berada 1–3% di bawahnya pada baseline (ε=∞), dan turun lebih dalam saat ε mengecil.